In [2]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.modeling import ModelingPreparer
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

#### Load e-commerce cleaned data

In [3]:
df_ecomm = pd.read_csv('../data/processed/ecommerce_processed.csv')
print(f"E-commerce data: {df_ecomm.shape}")
print(f"Columns: {df_ecomm.columns.tolist()[:10]}...")  

E-commerce data: (151112, 17)
Columns: ['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address']...


#### Prepare splits for e-commerce

In [9]:
preparer_ecomm = ModelingPreparer(target_col='class')
X_train, X_test, y_train, y_test = preparer_ecomm.prepare_splits(df_ecomm)

print("\n" + "="*60)
print("E-COMMERCE DATA SPLIT")
print("="*60)
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")
print(f"Test fraud rate: {y_test.mean()*100:.4f}%")


E-COMMERCE DATA SPLIT
Train shape: (120889, 191)
Test shape: (30223, 191)
Train fraud rate: 9.3648%
Test fraud rate: 9.3637%


#### Apply SMOTE with ratio (1:3)

In [10]:
X_train_balanced, y_train_balanced = preparer_ecomm.apply_resampling(
    X_train, y_train, 
    strategy='smote', 
    target_ratio=0.33  # 25% fraud, 1:3 ratio
)

##### Get and display imbalance report

In [11]:
report_ecomm = preparer_ecomm.get_imbalance_report()
print("\n" + "="*60)
print("CLASS DISTRIBUTION - E-COMMERCE")
print("="*60)
print(f"BEFORE SMOTE:")
print(f"  Legitimate: {report_ecomm['before']['legit']:,}")
print(f"  Fraud: {report_ecomm['before']['fraud']:,}")
print(f"  Ratio: {report_ecomm['before']['ratio']}")
print(f"\nAFTER SMOTE:")
print(f"  Legitimate: {report_ecomm['after']['legit']:,}")
print(f"  Fraud: {report_ecomm['after']['fraud']:,}")
print(f"  Ratio: {report_ecomm['after']['ratio']}")


CLASS DISTRIBUTION - E-COMMERCE
BEFORE SMOTE:
  Legitimate: 109,568
  Fraud: 11,321
  Ratio: 1:9.7

AFTER SMOTE:
  Legitimate: 109,568
  Fraud: 36,157
  Ratio: 1:3.0


### Load credit card cleaned data

In [5]:
df_credit = pd.read_csv('../data/processed/creditcard_cleaned.csv')
print(f"Credit card data: {df_credit.shape}")
print(f"Columns: {df_credit.columns.tolist()[:10]}...")

Credit card data: (283726, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']...


### Prepare splits for credit card

In [7]:
preparer_credit = ModelingPreparer(target_col='Class')  # Note: Capital 'C'
X_train_credit, X_test_credit, y_train_credit, y_test_credit = preparer_credit.prepare_splits(df_credit)
print("\n" + "="*60)
print("CREDIT CARD DATA SPLIT")
print("="*60)
print(f"Train shape: {X_train_credit.shape}")
print(f"Test shape: {X_test_credit.shape}")
print(f"Train fraud rate: {y_train_credit.mean()*100:.4f}%")
print(f"Test fraud rate: {y_test_credit.mean()*100:.4f}%")


CREDIT CARD DATA SPLIT
Train shape: (226980, 30)
Test shape: (56746, 30)
Train fraud rate: 0.1665%
Test fraud rate: 0.1674%


### Resampling Justification Documentation

In [8]:
# Cell: Resampling strategy justification
print("="*60)
print("RESAMPLING STRATEGY JUSTIFICATION")
print("="*60)

print("""
E-COMMERCE DATA (1:9.7 imbalance):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategy: SMOTE with target_ratio=0.33 (1:3 ratio)

Rationale:
1. Moderate imbalance (9.4% fraud) - SMOTE creates realistic synthetic samples
2. Rich feature space (device_id, browser, source, time features) allows good interpolation
3. 1:3 ratio prevents overfitting while giving model enough fraud examples
4. SMOTE preferred over simple oversampling to avoid duplicate samples

CREDIT CARD DATA (1:599 imbalance):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategy: SMOTE with target_ratio=0.1 (1:10 ratio) OR Combined approach

Rationale:
1. Extreme imbalance requires aggressive resampling
2. Pure undersampling would discard 99.8% of data - unacceptable
3. Pure oversampling would cause severe overfitting
4. SMOTE with conservative ratio (1:10) balances learning
5. Alternative: SMOTE-ENN (SMOTE + Edited Nearest Neighbors) for noisy samples

IMPLEMENTATION NOTE:
Always apply resampling ONLY on training set after train-test split
to prevent data leakage.
""")

RESAMPLING STRATEGY JUSTIFICATION

E-COMMERCE DATA (1:9.7 imbalance):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategy: SMOTE with target_ratio=0.33 (1:3 ratio)

Rationale:
1. Moderate imbalance (9.4% fraud) - SMOTE creates realistic synthetic samples
2. Rich feature space (device_id, browser, source, time features) allows good interpolation
3. 1:3 ratio prevents overfitting while giving model enough fraud examples
4. SMOTE preferred over simple oversampling to avoid duplicate samples

CREDIT CARD DATA (1:599 imbalance):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Strategy: SMOTE with target_ratio=0.1 (1:10 ratio) OR Combined approach

Rationale:
1. Extreme imbalance requires aggressive resampling
2. Pure undersampling would discard 99.8% of data - unacceptable
3. Pure oversampling would cause severe overfitting
4. SMOTE with conservative ratio (1:10) balances learning
5. Alternative: SMOTE-ENN (SMOTE + Edited Nearest Neighbors) for noisy samples

IMPLEMENTATION N